In [72]:
import numpy as np
import pandas as pd
import cv2
import ultralytics

In [73]:
yolo_model=ultralytics.YOLO('best1.pt')

In [74]:
img='image.png'
result=yolo_model(img)


image 1/1 c:\Users\sreer\OneDrive\Desktop\VisionFabric\image.png: 640x448 1 trousers, 140.4ms
Speed: 4.0ms preprocess, 140.4ms inference, 1.4ms postprocess per image at shape (1, 3, 640, 448)


In [75]:
# for r in result:
#     img = r.orig_img.copy()

#     for box in r.boxes:
#         x1, y1, x2, y2 = map(int, box.xyxy[0])

#         cls_id = int(box.cls[0])        # class index
#         name = r.names[cls_id]          # class name

#         # draw box
#         cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), 2)

#         # put label
#         cv2.putText(img, name,
#                     (x1, y1 - 10),
#                     cv2.FONT_HERSHEY_SIMPLEX,
#                     0.7,
#                     (0, 255, 0),
#                     2)
#         print(name)

#     cv2.imshow("frame", img)
#     cv2.waitKey(0)
#     cv2.destroyAllWindows()

In [76]:
def detect_dress(img_rgb):
    results = yolo_model(img_rgb)
    boxes = []
    names = []
    
    for r in results:
        for box in r.boxes:
            # Get coordinates
            b = box.xyxy[0].cpu().numpy().astype(int)
            boxes.append(b)
            # Get label name
            names.append(yolo_model.names[int(box.cls)])
            
    return boxes, names

# # 1. Load the image
# img_bgr = cv2.imread('image.png')

# # 2. Convert BGR to RGB (CRITICAL STEP)
# img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

# # 3. Now run the detection
# boxes, names = detect_dress(img_rgb)

# print(f"Detected {len(boxes)} items: {names}")

In [77]:
# img='image.png'
# img=cv2.imread(img)
# cv2.imshow('frame',img)
# cv2.waitKey(3000)
# cv2.destroyAllWindows()

In [78]:
import torch
import cv2
import numpy as np
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

# 1. SETUP (Load once)
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# 2. DEFINE ATTRIBUTES (The "Knowledge Base" for CLIP)
# You can add or change these anytime without retraining!
attributes = {
    "color": ["black", "white", "gray", "red", "blue","green", "yellow", "brown", "pink", "purple"],
    "fit": ["oversized baggy fit", "slim tight fit", "regular standard fit", "cropped short fit"],
    "style": ["formal professional", "casual streetwear", "vintage retro", "sporty athletic"]
    
}

def get_best_attribute(crop_pil, category_name, options):
    """Helper to ask CLIP to pick the best option from a list"""
    # We create descriptive prompts like: "a photo of a [slim tight fit] garment"
    prompts = [f"a photo of a {opt} garment" for opt in options]
    # print(prompts)
    inputs = clip_processor(text=prompts, images=crop_pil, return_tensors="pt", padding=True).to(device)
    
    with torch.no_grad():
        outputs = clip_model(**inputs)
    
    probs = outputs.logits_per_image.softmax(dim=1)
    best_idx = probs.argmax().item()
    return options[best_idx], probs[0][best_idx].item()

def smart_detect(img_array):
    # Get the "Where" and the basic "Name" from your YOLO
    boxes, names = detect_dress(img_array) 
    print(boxes,names)
    analysis_results = []

    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = box
        yolo_name = names[i] # e.g., "shirt"
        
        # Crop and prepare image
        crop = img_array[y1:y2, x1:x2]
        if crop.size == 0: continue
        crop_pil = Image.fromarray(crop)
        
        # Extract features one by one
        features = {"category": yolo_name, "box": box}
        
        for attr_type, options in attributes.items():
            best_val, confidence = get_best_attribute(crop_pil, attr_type, options)
            features[attr_type] = best_val
            
        # Create a "RAG String" - This is what you send to your LLM
        rag_description = (
            f"{features['fit']} {features['color']} "
            f"{features['category']}  in a {features['style']} style"
        )
        features["description"] = rag_description
        
        analysis_results.append(features)
        
    return analysis_results



Loading weights: 100%|██████████| 398/398 [00:00<00:00, 36131.97it/s]


In [79]:
# --- HOW TO RUN ---
raw_img = cv2.imread('img.png')
ready_img = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)
results = smart_detect(ready_img)

for item in results:
    print(f"Detected: {item['description']}")


0: 640x512 1 long_sleeved_shirt, 1 trousers, 128.7ms
Speed: 2.9ms preprocess, 128.7ms inference, 2.5ms postprocess per image at shape (1, 3, 640, 512)
[array([ 418,  648,  745, 1225]), array([346, 188, 778, 730])] ['trousers', 'long_sleeved_shirt']
Detected: slim tight fit white trousers  in a sporty athletic style
Detected: regular standard fit blue long_sleeved_shirt  in a formal professional style


In [80]:
print(results)

[{'category': 'trousers', 'box': array([ 418,  648,  745, 1225]), 'color': 'white', 'fit': 'slim tight fit', 'style': 'sporty athletic', 'description': 'slim tight fit white trousers  in a sporty athletic style'}, {'category': 'long_sleeved_shirt', 'box': array([346, 188, 778, 730]), 'color': 'blue', 'fit': 'regular standard fit', 'style': 'formal professional', 'description': 'regular standard fit blue long_sleeved_shirt  in a formal professional style'}]


In [81]:
# outfit_features = " & ".join(item['description'] for item in results) + "."
# print(outfit_features)

In [82]:
# RAG implementataion
import chromadb
import ollama

In [83]:
client=chromadb.PersistentClient('./my_local_data')

In [84]:
collection=client.get_or_create_collection(name='my_knowledge_base')

In [85]:
# Clean the current_outfit string to extract core features
outfit_features = " & ".join(item['description'] for item in results) + "."
occasion_type = "Casual/Lifestyle"

# Construct a high-density query
query = f"Occasion: {occasion_type} | Current: User is wearing a  {outfit_features} | Recommendation: accessories, footwear, styling improvements"

results = collection.query(
    query_texts=[query],
    n_results=3
)

In [86]:
current_outfit="""User is wearing a slim tight fit white trousers made of linen,User is wearing a regular standard fit blue long_sleeved_shirt made of denim rugged material."""
occasion="Casual/Lifestyle"

prompt=f"""
### Role
You are a Senior Fashion Stylist. Your goal is to adapt the user's current outfit to a specific occasion using expert style logic, with a high-level focus on color theory and garment construction.

### Contextual Style Guidelines
{results['documents'][0]}

### User Input
- **Current Outfit:** {current_outfit}
- **Target Occasion:** {occasion}

### Instructions
1. **The Adaptation:** Analyze how the "Current Outfit" can be elevated to fit the "Target Occasion." Apply the "Color Strategy" and "Guidelines" from the context to bridge the gap between the user's current pieces and the desired aesthetic.
2. **Color Combination Logic:** - Evaluate the interaction between the existing colors (e.g., the contrast between the white linen and blue denim).
    - Suggest a specific color palette for added layers or accessories that follows the "Color Strategy" (e.g., Monochromatic, Complementary, or Analogous) to harmonize the look.
3. **Styling Fixes:** Provide specific technical advice on how to wear the current pieces (e.g., precise tucking methods, sleeve rolls, or structural layering) to meet the standards of the occasion.
4. **Accessory Integration:** Recommend specific accessories (footwear, belts, timepieces) that use color and texture to transition the casual materials (denim/linen) into a sophisticated "Casual/Lifestyle" ensemble.

### Constraints
- Focus strictly on the "Target Occasion" provided.
- Maintain a sophisticated, authoritative, and expert tone.
- Do not include conversational filler or AI self-references.
- Output the recommendation in clear, concise bullet points using professional fashion terminology.
"""

In [87]:
output=ollama.generate(model="gemma3:1b",prompt=prompt,options={"temperature":0.1})

In [88]:
print(output['response'])

Okay, here’s a detailed adaptation of your outfit for the Casual/Lifestyle occasion, incorporating the provided context and guidelines:

**Recommendation:**

The current outfit presents a solid base, but it lacks the sophistication required for a Casual/Lifestyle event. We’ll elevate it through strategic color and textural adjustments.

**1. Color Strategy & Harmony:**

*   **Current Colors:** The white linen trousers and blue denim create a relatively neutral base. However, the blue denim’s intensity contrasts with the white, creating a slight imbalance.
*   **Proposed Color Palette:** We’ll lean towards a **Monochromatic** approach, utilizing shades of blue and grey. Specifically, consider a deep teal or a muted indigo as a base, paired with a lighter, sandy beige or a soft grey for the linen trousers. This creates a cohesive and subtly stylish look.  Avoid bright, saturated blues – we’re aiming for a calming, sophisticated feel.

**2. Styling Fixes & Technical Advice:**

*   **Tuck 